In [35]:
# Question 1: What rows and columns are in the dataset?

import csv
from pathlib import Path
from typing import Any
import pandas as pd


DATA_PATH = Path("data") / "electronics_sales.csv"
OUTPUT_PATH = Path("output") / "sales_report.txt"


def parse_number(value: str | None) -> float | None:
    """Convert a messy numeric string to a float or return None."""
    if value is None:
        return None

    cleaned = str(value).strip()
    if not cleaned:
        return None

    cleaned = cleaned.upper().replace("KES", "").replace(",", "").strip()

    if cleaned in {"", "N/A", "NONE", "NULL", "-"}:
        return None

    try:
        return float(cleaned)
    except ValueError:
        return None


def read_sales(path: Path = DATA_PATH) -> list[dict[str, str]]:
    """Read the CSV file and return the raw rows."""
    with open(path, mode="r", newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        rows = list(reader)

    print(f"Number of rows found: {len(rows)}")
    print(f"Column names found: {reader.fieldnames}")
    return rows


sample_rows = read_sales(DATA_PATH)
print(f"Sample keys: {list(sample_rows[0].keys()) if sample_rows else []}")


Number of rows found: 50
Column names found: ['sale_id', 'sale_date', 'store', 'product', 'quantity', 'unit_price_kes', 'payment_method']
Sample keys: ['sale_id', 'sale_date', 'store', 'product', 'quantity', 'unit_price_kes', 'payment_method']


In [36]:
# Question 2: How do we clean the messy numeric values?


def find_value(row: dict[str, str], *names: str) -> str | None:
    """Find a field value even when the key names are slightly varied."""
    normalized = {
        key.lower().replace("_", "").replace(" ", ""): value
        for key, value in row.items()
    }

    for name in names:
        target = name.lower().replace("_", "").replace(" ", "")
        if target in normalized:
            return normalized[target]

        for key, value in normalized.items():
            if target in key:
                return value

    return None


def convert_rows(rows: list[dict[str, str]]) -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    """Clean valid rows and keep invalid rows with a reason."""
    clean_rows: list[dict[str, Any]] = []
    skipped_rows: list[dict[str, Any]] = []

    for row_number, row in enumerate(rows, start=1):
        quantity_raw = find_value(row, "quantity", "qty")
        unit_price_raw = find_value(row, "unit_price", "unit price", "price")
        revenue_raw = find_value(row, "revenue", "total sales", "sales")

        quantity = parse_number(quantity_raw)
        unit_price = parse_number(unit_price_raw)
        revenue: float | None = parse_number(revenue_raw) if revenue_raw is not None else None

        if quantity is None:
            skipped_rows.append({
                "row": row_number,
                "sale_id": row.get("sale_id"),
                "reason": "invalid quantity",
                "data": row,
            })
            continue

        if unit_price is None:
            skipped_rows.append({
                "row": row_number,
                "sale_id": row.get("sale_id"),
                "reason": "invalid unit price",
                "data": row,
            })
            continue

        if quantity <= 0 or unit_price <= 0:
            skipped_rows.append({
                "row": row_number,
                "sale_id": row.get("sale_id"),
                "reason": "non positive value",
                "data": row,
            })
            continue

        if revenue_raw is not None and str(revenue_raw).strip() and revenue is None:
            skipped_rows.append({
                "row": row_number,
                "sale_id": row.get("sale_id"),
                "reason": "invalid revenue",
                "data": row,
            })
            continue

        if revenue is None:
            revenue = quantity * unit_price

        clean_row = dict(row)
        clean_row["quantity"] = quantity
        clean_row["unit_price"] = unit_price
        clean_row["revenue"] = revenue
        clean_rows.append(clean_row)

    return clean_rows, skipped_rows


sample_clean, sample_skipped = convert_rows(read_sales(DATA_PATH))
print(f"Clean rows: {len(sample_clean)}")
print(f"Skipped rows: {len(sample_skipped)}")
print("First cleaned row:", sample_clean[0] if sample_clean else None)


Number of rows found: 50
Column names found: ['sale_id', 'sale_date', 'store', 'product', 'quantity', 'unit_price_kes', 'payment_method']
Clean rows: 46
Skipped rows: 4
First cleaned row: {'sale_id': 'S-3001', 'sale_date': '2026-09-01', 'store': 'Westgate', 'product': 'Safaricom Neon Ray', 'quantity': 1.0, 'unit_price_kes': '5999', 'payment_method': 'Cash', 'unit_price': 5999.0, 'revenue': 5999.0}


In [37]:
# Question 3: How much revenue is generated by each grouping?


def total_revenue(rows: list[dict[str, Any]]) -> float:
    """Calculate total revenue for all valid rows."""
    total = 0.0
    for row in rows:
        revenue_value = row.get("revenue")
        if revenue_value is None:
            quantity = parse_number(row.get("quantity"))
            unit_price = parse_number(row.get("unit_price"))
            if quantity is not None and unit_price is not None:
                revenue_value = quantity * unit_price

        if revenue_value is not None:
            total += float(revenue_value)

    return total


def revenue_by(rows: list[dict[str, Any]], field: str) -> dict[str, float]:
    """Group revenue by a field such as product or store."""
    totals: dict[str, float] = {}

    for row in rows:
        key = row.get(field)
        if key is None:
            continue

        amount = float(row.get("revenue", 0.0))
        totals[key] = totals.get(key, 0.0) + amount

    return totals


clean_rows, _ = convert_rows(read_sales(DATA_PATH))
total = total_revenue(clean_rows)
product_totals = revenue_by(clean_rows, "product")
print(f"Total revenue: {total:,.2f}")
print("Top product totals:", dict(sorted(product_totals.items(), key=lambda item: (-item[1], item[0].lower()))[:3]))


Number of rows found: 50
Column names found: ['sale_id', 'sale_date', 'store', 'product', 'quantity', 'unit_price_kes', 'payment_method']
Total revenue: 1,310,812.00
Top product totals: {'HP 15 Laptop': 389994.0, 'Hisense 43in Smart TV': 272993.0, 'Lenovo Tab M10': 199992.0}


In [38]:
# Question 4: Which products have the highest revenue?


def top_n(totals: dict[str, float], n: int = 3) -> list[tuple[str, float]]:
    """Return the highest revenue items with a deterministic tie-break."""
    ranked = sorted(totals.items(), key=lambda item: (-item[1], item[0].lower()))
    return ranked[:n]


def high_value(rows: list[dict[str, Any]], threshold: float = 50000.0) -> list[dict[str, Any]]:
    """Return transactions whose revenue meets the high-value threshold."""
    return [row for row in rows if float(row.get("revenue", 0.0)) >= threshold]


product_totals = revenue_by(clean_rows, "product")
print("Top 3 products:", top_n(product_totals, 3))
print("High-value transactions:", len(high_value(clean_rows, 50000.0)))


Top 3 products: [('HP 15 Laptop', 389994.0), ('Hisense 43in Smart TV', 272993.0), ('Lenovo Tab M10', 199992.0)]
High-value transactions: 5


In [39]:
# Question 5: What are the transaction statistics?


def mean(values: list[float]) -> float:
    """Return the arithmetic mean for a non-empty list."""
    if not values:
        raise ValueError("Values cannot be empty")
    return sum(values) / len(values)


def median(values: list[float]) -> float:
    """Return the median for a non-empty list."""
    if not values:
        raise ValueError("Values cannot be empty")

    ordered = sorted(values)
    midpoint = len(ordered) // 2

    if len(ordered) % 2 == 0:
        return (ordered[midpoint - 1] + ordered[midpoint]) / 2

    return ordered[midpoint]


def minimum(values: list[float]) -> float:
    """Return the smallest value in a non-empty list."""
    if not values:
        raise ValueError("Values cannot be empty")
    return min(values)


def maximum(values: list[float]) -> float:
    """Return the largest value in a non-empty list."""
    if not values:
        raise ValueError("Values cannot be empty")
    return max(values)


def transaction_statistics(rows: list[dict[str, Any]]) -> dict[str, float]:
    """Calculate summary statistics for transaction revenue."""
    revenue_values = [float(row["revenue"]) for row in rows]
    return {
        "mean": mean(revenue_values),
        "median": median(revenue_values),
        "min": minimum(revenue_values),
        "max": maximum(revenue_values),
    }


stats = transaction_statistics(clean_rows)
print("Revenue statistics:", stats)


Revenue statistics: {'mean': 28495.91304347826, 'median': 11097.0, 'min': 1499.0, 'max': 259996.0}


In [40]:
# Question 6: Does pandas confirm the result?


def verify_with_pandas(
    path: Path = DATA_PATH,
    my_results: dict[str, Any] | None = None,
    print_results: bool = True,
) -> dict[str, Any]:
    """Verify the custom result set against pandas-based calculations."""
    if pd is None:
        raise ImportError("pandas is required for verification")

    dataframe = pd.read_csv(path)
    dataframe = dataframe.copy()
    dataframe["quantity"] = dataframe["quantity"].map(parse_number)
    dataframe["unit_price_kes"] = dataframe["unit_price_kes"].map(parse_number)
    dataframe = dataframe.dropna(subset=["quantity", "unit_price_kes"])
    dataframe = dataframe[dataframe["quantity"] > 0]
    dataframe = dataframe[dataframe["unit_price_kes"] > 0]
    dataframe["revenue"] = dataframe["quantity"] * dataframe["unit_price_kes"]

    product_totals = dataframe.groupby("product")["revenue"].sum().sort_values(ascending=False)
    pandas_totals = product_totals.to_dict()
    pandas_top = list(product_totals.head(3).items())
    stats = transaction_statistics([
        {"revenue": float(value)} for value in dataframe["revenue"].tolist()
    ])

    if my_results is None:
        raw_rows = read_sales(path)
        clean_rows, _ = convert_rows(raw_rows)
        my_results = {
            "total_revenue": total_revenue(clean_rows),
            "product_totals": revenue_by(clean_rows, "product"),
            "top_3": top_n(revenue_by(clean_rows, "product"), 3),
            "stats": transaction_statistics(clean_rows),
        }

    total_match = round(my_results.get("total_revenue", 0.0), 2) == round(float(dataframe["revenue"].sum()), 2)
    product_match = my_results.get("product_totals", {}) == pandas_totals
    top_match = my_results.get("top_3", []) == pandas_top
    stats_match = my_results.get("stats", {}) == stats

    if print_results:
        print("=== pandas verification ===")
        print(f"Total revenue      : {'MATCH' if total_match else 'MISMATCH'}")
        print(f"Revenue by product : {'MATCH' if product_match else 'MISMATCH'}")
        print(f"Top 3              : {'MATCH' if top_match else 'MISMATCH'}")
        print(f"Mean / median      : {'MATCH' if stats_match else 'MISMATCH'}")

    return {
        "total_revenue": total_match,
        "product_totals": product_match,
        "top_3": top_match,
        "stats": stats_match,
    }


verification = verify_with_pandas(DATA_PATH, {
    "total_revenue": total_revenue(clean_rows),
    "product_totals": revenue_by(clean_rows, "product"),
    "top_3": top_n(revenue_by(clean_rows, "product"), 3),
    "stats": transaction_statistics(clean_rows),
}, print_results=True)
print("Verification summary:", verification)


=== pandas verification ===
Total revenue      : MATCH
Revenue by product : MATCH
Top 3              : MATCH
Mean / median      : MATCH
Verification summary: {'total_revenue': True, 'product_totals': True, 'top_3': True, 'stats': True}


In [41]:
# Question 7: How do we write the final report?


def build_report(rows: list[dict[str, Any]], skipped_rows: list[dict[str, Any]]) -> str:
    """Generate the final text report for the project output."""
    total = total_revenue(rows)
    product_totals = revenue_by(rows, "product")
    top_products = top_n(product_totals, 3)
    qualifying = high_value(rows, threshold=50000.0)
    stats = transaction_statistics(rows)

    skip_summary = ", ".join(
        f"{entry.get('sale_id', '?')}: {entry.get('reason', 'unknown')}"
        for entry in skipped_rows
    )

    lines = [
        "=== Sales report: September 2026 ===",
        f"Rows read          : {len(rows) + len(skipped_rows)}",
        f"Rows used          : {len(rows)}",
        f"Rows skipped       : {len(skipped_rows)}   ({skip_summary})",
        "",
        f"Total revenue (KES): {total:,.2f}",
        "",
        "Revenue by product (KES)",
    ]

    for product, revenue in sorted(product_totals.items(), key=lambda item: (-item[1], item[0].lower())):
        lines.append(f"  {product:<23} {revenue:,.2f}")

    lines.extend(["", "Top 3 products"])

    for rank, (product, revenue) in enumerate(top_products, start=1):
        lines.append(f"  {rank}. {product} - KES {revenue:,.2f}")

    lines.extend([
        "",
        f"High-value transactions (>= KES 50,000): {len(qualifying)}",
    ])

    for row in qualifying:
        lines.append(
            f"  {row.get('sale_id', '?')}  {row.get('sale_date', '?')}  {row.get('store', '?')}  {row.get('product', '?')}  KES {float(row.get('revenue', 0.0)):,.2f}"
        )

    lines.extend([
        "",
        "Transaction revenue statistics",
        f"  mean   : {stats['mean']:,.2f}",
        f"  median : {stats['median']:,.2f}",
        f"  min    : {stats['min']:,.2f}",
        f"  max    : {stats['max']:,.2f}",
        "",
        "=== pandas verification ===",
    ])

    verification = verify_with_pandas(DATA_PATH, {
        "total_revenue": total,
        "product_totals": product_totals,
        "top_3": top_products,
        "stats": stats,
    }, print_results=False)

    lines.append(f"Total revenue      : {'MATCH' if verification['total_revenue'] else 'MISMATCH'}")
    lines.append(f"Revenue by product : {'MATCH' if verification['product_totals'] else 'MISMATCH'}")
    lines.append(f"Top 3              : {'MATCH' if verification['top_3'] else 'MISMATCH'}")
    lines.append(f"Mean / median      : {'MATCH' if verification['stats'] else 'MISMATCH'}")

    return "\n".join(lines) + "\n"


def main() -> None:
    """Run the sales analysis and save the report to the output folder."""
    raw_rows = read_sales(DATA_PATH)
    clean_rows, skipped_rows = convert_rows(raw_rows)
    report_text = build_report(clean_rows, skipped_rows)

    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    OUTPUT_PATH.write_text(report_text, encoding="utf-8")
    print(report_text)


if __name__ == "__main__":
    main()


Number of rows found: 50
Column names found: ['sale_id', 'sale_date', 'store', 'product', 'quantity', 'unit_price_kes', 'payment_method']
=== Sales report: September 2026 ===
Rows read          : 50
Rows used          : 46
Rows skipped       : 4   (S-3006: invalid quantity, S-3030: invalid unit price, S-3035: invalid quantity, S-3039: invalid unit price)

Total revenue (KES): 1,310,812.00

Revenue by product (KES)
  HP 15 Laptop            389,994.00
  Hisense 43in Smart TV   272,993.00
  Lenovo Tab M10          199,992.00
  Tecno Spark 20          161,988.00
  Infinix Hot 40          63,996.00
  Samsung Galaxy A15      56,997.00
  JBL Go 3 Speaker        49,491.00
  Anker Power Bank 20000  38,691.00
  Xiaomi Redmi Buds       25,592.00
  Oraimo FreePods         19,593.00
  USB-C Charger 25W       19,487.00
  Safaricom Neon Ray      11,998.00

Top 3 products
  1. HP 15 Laptop - KES 389,994.00
  2. Hisense 43in Smart TV - KES 272,993.00
  3. Lenovo Tab M10 - KES 199,992.00

High-value tr